# Convert SPoRT-LIS template


Author: Kyle Lesinger 

In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
import rioxarray as rxr
import s3fs
import subprocess
import fsspec
from rasterio.warp import calculate_default_transform, reproject, Resampling
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")

✅ Libraries imported successfully!
Boto3 version: 1.37.3


In [2]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog,
    export_COG_PROFILE
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Memory monitoring utilities loaded
✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


# Useful links
<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/" style="color: blue; font-size: 20px;">drcs_activations OLD Directory</a> -- You can view old directory file structure here.

<a href="https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html" style="color: blue; font-size: 20px;">VEDA docs for file naming conventions</a> -- Helps for understanding why/how we name content.

In [3]:
# DO NOT CHANGE
DIR_OLD_BASE = 'sport-lis'
DIR_NEW_BASE = 'nrt/sport-lis-cog' 
BUCKET = 'nasa-disasters'

In [4]:
EVENT_NAME = 'rsm0-2m'  #find the name within drcs_activations OLD Directory (see link above)

PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}'  # Updated to use actual available directory
DIRECTORY_NEW = f'{DIR_NEW_BASE}/{EVENT_NAME}'

In [5]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = export_COG_PROFILE()


## Initialize AWS S3 Client with automatic credential detection

In [6]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, BUCKET, PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

✅ S3 client initialized successfully
   Found 68 accessible buckets
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files
✅ Found 208 .tif files in the S3 bucket.


['sport-lis/rsm0-2m/20250217_0000_sport_lis_rsm0-2m_conus3km_float_wgs84.tif',
 'sport-lis/rsm0-2m/20250218_0000_sport_lis_rsm0-2m_conus3km_float_wgs84.tif',
 'sport-lis/rsm0-2m/20250219_0000_sport_lis_rsm0-2m_conus3km_float_wgs84.tif',
 'sport-lis/rsm0-2m/20250220_0000_sport_lis_rsm0-2m_conus3km_float_wgs84.tif',
 'sport-lis/rsm0-2m/20250221_0000_sport_lis_rsm0-2m_conus3km_float_wgs84.tif',
 'sport-lis/rsm0-2m/20250222_0000_sport_lis_rsm0-2m_conus3km_float_wgs84.tif',
 'sport-lis/rsm0-2m/20250223_0000_sport_lis_rsm0-2m_conus3km_float_wgs84.tif',
 'sport-lis/rsm0-2m/20250224_0000_sport_lis_rsm0-2m_conus3km_float_wgs84.tif',
 'sport-lis/rsm0-2m/20250225_0000_sport_lis_rsm0-2m_conus3km_float_wgs84.tif',
 'sport-lis/rsm0-2m/20250226_0000_sport_lis_rsm0-2m_conus3km_float_wgs84.tif',
 'sport-lis/rsm0-2m/20250227_0000_sport_lis_rsm0-2m_conus3km_float_wgs84.tif',
 'sport-lis/rsm0-2m/20250228_0000_sport_lis_rsm0-2m_conus3km_float_wgs84.tif',
 'sport-lis/rsm0-2m/20250301_0000_sport_lis_rsm0-2m_

# Useful links
<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/" style="color: blue; font-size: 20px;">drcs_activations OLD Directory</a> -- You can view old directory file structure here.

<a href="https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html" style="color: blue; font-size: 20px;">VEDA docs for file naming conventions</a> -- Helps for understanding why/how we name content.

In [7]:

config = {
    "data_acquisition_method": "s3",
    "raw_data_bucket" : BUCKET, #DO NOT CHANGE
    "raw_data_prefix": PATH_OLD,
    "cog_data_bucket": BUCKET, #DO NOT CHANGE
    "cog_data_prefix": f"{DIRECTORY_NEW}",  #We changed this!!!!!!!
    "local_output_dir": f"output/{EVENT_NAME}",  # Local directory to save COGs
    "transformation": {}
}

def add_year_to_config(base_config,year):
    import copy
    config = copy.deepcopy(base_config)
    config['cog_data_prefix'] = f'{config['cog_data_prefix']}/{year}'
    return config
    

add_year_to_config(config,2022)

{'data_acquisition_method': 's3',
 'raw_data_bucket': 'nasa-disasters',
 'raw_data_prefix': 'sport-lis/rsm0-2m',
 'cog_data_bucket': 'nasa-disasters',
 'cog_data_prefix': 'nrt/sport-lis-cog/rsm0-2m/2022',
 'local_output_dir': 'output/rsm0-2m',
 'transformation': {}}

In [8]:
def return_bucket_info(config,year):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = f'{config["cog_data_prefix"]}/{year}'
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }




In [9]:
# Use the function to test what the output config will be 
_bucket = return_bucket_info(config, 2022)


Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: sport-lis/rsm0-2m
  Target bucket: nasa-disasters
  Target prefix: nrt/sport-lis-cog/rsm0-2m/2022


## Define COG Conversion Function

This function handles the conversion of files to Cloud Optimized GeoTIFFs with proper CRS and caching.

In [10]:
def cog_validation_checker(input_filename):
    output = subprocess.run(["rio","cogeo", "validate", input_filename],capture_output=True, text=True)
    cogValid = False if "NOT a valid cloud optimized GeoTIFF" in output.stdout else True
    return cogValid

In [11]:
def convert_to_proper_CRS_and_cogify(name, cog_filename, cog_data_bucket, cog_data_prefix, year, local_output_dir=None):
    """
    Convert a file to Cloud Optimized GeoTIFF with proper CRS.
    
    This function includes:
    - Download caching to avoid re-downloading files
    - CRS reprojection to EPSG:4326
    - COG validation before upload
    - Upload to S3
    - Smart nodata value handling based on data type
    """
    s3_key = f"{cog_data_prefix}/{cog_filename}"
    reproject_filename = f"reproj/{cog_filename}"
    
    # Create necessary directories
    os.makedirs("reproj", exist_ok=True)
    
    # Create data_download directory for caching
    data_download_dir = "data_download"
    os.makedirs(data_download_dir, exist_ok=True)
    '''
    # Create subdirectory structure to match S3 path
    s3_path_parts = name.split('/')
    local_subdir = os.path.join(data_download_dir, *s3_path_parts[:-1])
    os.makedirs(local_subdir, exist_ok=True)
    
    # Local path for the downloaded file (persistent storage)
    local_download_path = os.path.join(data_download_dir, name)
    '''
    local_download_path = f"data_download/{name.split("/")[-1]}"

    # Temporary file for processing
    temp_input_file = f"temp_{os.path.basename(name)}"

    try:

        # Check if file already exists locally
        if os.path.exists(local_download_path):
            print(f"   [CACHE HIT] Using cached file: {local_download_path}")
            import shutil
            shutil.copy(local_download_path, temp_input_file)
        else:
            # Download the file from S3
            print(f"   [DOWNLOAD] Downloading from S3...")
            s3_client.download_file(BUCKET, name, local_download_path)
            print(f"   [DOWNLOAD] ✅ Saved to cache")
            import shutil
            shutil.copy(local_download_path, temp_input_file)
        
        # Reproject to EPSG:4326
        print(f"   [REPROJECT] Converting to EPSG:4326...")
        with rasterio.open(temp_input_file) as src:
            dst_crs = "EPSG:4326"
            
            # Check if reprojection is needed
            if src.crs and src.crs.to_string() == dst_crs:
                print(f"   [REPROJECT] Already in {dst_crs}, skipping reprojection")
                import shutil
                shutil.copy(temp_input_file, reproject_filename)
            else:
                transform, width, height = calculate_default_transform(
                    src.crs, dst_crs, src.width, src.height, *src.bounds
                )
                kwargs = src.meta.copy()
                kwargs.update({
                    "driver": "COG",
                    "compress": "DEFLATE",
                    "crs": dst_crs,
                    "transform": transform,
                    "width": width,
                    "height": height
                })

                with rasterio.open(reproject_filename, "w", **kwargs) as dst:
                    for band_idx in range(1, src.count + 1):
                        reproject(
                            source=rasterio.band(src, band_idx),
                            destination=rasterio.band(dst, band_idx),
                            src_transform=src.transform,
                            src_crs=src.crs,
                            dst_transform=transform,
                            dst_crs=dst_crs,
                            resampling=Resampling.nearest,
                            wrapdateline=True
                        )

        # COGify & upload
        print(f"   [COGIFY] Creating COG...")
        ds = rxr.open_rasterio(reproject_filename)
        
        # Handle coordinate naming
        if "y" in ds.dims and "x" in ds.dims:
            ds = ds.rename({"y": "lat", "x": "lon"})
            ds.rio.set_spatial_dims("lon", "lat", inplace=True)
        
        # Smart nodata value handling based on data type
        print(f"   [NODATA] Data type: {ds.dtype}")
        if ds.dtype == 'uint8':
            # For RGB images (uint8), use 0 as nodata (black pixels)
            nodata_value = 0
            print(f"   [NODATA] Using nodata value {nodata_value} for uint8 data")
        elif ds.dtype == 'uint16':
            # For uint16, use 0 as nodata
            nodata_value = 0
            print(f"   [NODATA] Using nodata value {nodata_value} for uint16 data")
        else:
            # For float32, int16, int32, etc., use -9999
            nodata_value = -9999
            print(f"   [NODATA] Using nodata value {nodata_value} for {ds.dtype} data")
        
        ds.rio.write_nodata(nodata_value, inplace=True)

        with tempfile.NamedTemporaryFile(suffix='.tif', delete=False) as tmp:
            tmp_name = tmp.name
            ds.rio.to_raster(tmp_name, **COG_PROFILE)
            
            # Validate COG
            print(f"   [VALIDATE] Checking COG validity...")
            is_valid_cog, validation_details = validate_cog(tmp_name)
            
            if is_valid_cog:
                print(f"   [VALIDATE] ✅ Valid COG")
            else:
                print(f"   [VALIDATE] ⚠️ COG validation warnings")
                critical_errors = [e for e in validation_details['errors'] if 'Invalid driver' in e]
                if critical_errors:
                    raise ValueError(f"Critical COG validation failed")
                if 'errors' in validation_details:
                    for error in validation_details['errors']:
                        print(f"      - {error}")
                if 'warnings' in validation_details:
                    for warning in validation_details['warnings']:
                        print(f"      - {warning}")
            
            # Upload to S3
            print(f"   [UPLOAD] Uploading to S3...")
            s3_client.upload_file(
                Filename=tmp_name,
                Bucket=cog_data_bucket,
                Key=s3_key
            )
            print(f"   [SUCCESS] ✅ Uploaded to s3://{cog_data_bucket}/{s3_key}")
            
            # Save locally if specified
            if local_output_dir:
                print("were in here")
                os.makedirs(local_output_dir, exist_ok=True)
                local_path = os.path.join(local_output_dir, cog_filename)
                print(local_path)
                import shutil
                shutil.copy(tmp_name, local_path)
    
    except Exception as e:
        print(f"   [ERROR] Failed: {str(e)}")
        raise
            
    finally:
        # Clean up temporary files
        for temp_file in [temp_input_file, reproject_filename]:
            if os.path.exists(temp_file):
                os.remove(temp_file)
        if 'tmp_name' in locals() and os.path.exists(tmp_name):
            os.remove(tmp_name)

        original_s3_uri = f"s3://{BUCKET}/{name}"
        target_s3_uri = f"s3://{cog_data_bucket}/{cog_data_prefix}/{cog_filename}"
        oldCog_status = cog_validation_checker(original_s3_uri)
        newCog_status = cog_validation_checker(target_s3_uri)
        print(f"Old: {oldCog_status}, New: {newCog_status}")
        if not oldCog_status and not newCog_status:
            raise Exception("An error has occurred in the conversion. Stopping the execution of the script")
        elif newCog_status:
            os.remove(local_download_path)
            
print("✅ COG conversion function defined with smart nodata handling")

✅ COG conversion function defined with smart nodata handling


In [12]:
def prepare_cogification(name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir):
    """               
    Convert a file to Cloud Optimized GeoTIFF with proper CRS.
    
    This function includes:
    - creating key names
    - determine cog processing
    """
    original_s3_uri = f"s3://{BUCKET}/{name}"
    target_s3_uri = f"s3://{cog_data_bucket}/{cog_data_prefix}/{cog_filename}"

    s3_client = boto3.client('s3')
    try:
        s3_client.head_object(Bucket=cog_data_bucket, Key=f"{cog_data_prefix}/{cog_filename}")
        newCog_exists = True
    except:
        newCog_exists = False

    oldCog_status = cog_validation_checker(original_s3_uri)
    newCog_status = False if not newCog_exists else cog_validation_checker(target_s3_uri)
    
    print(f"Old: {oldCog_status}, New: {newCog_status}")
    
    if newCog_status:
        try:
            filename = name.split("/")[-1]
            os.remove(f"data_download/{filename}")
            os.remove(f"output/{filename}")
            os.remove(f"reproj/{filename}")
        except:
            print("no files to delete")
    elif not oldCog_status and not newCog_status:
        convert_to_proper_CRS_and_cogify(name, cog_filename, cog_data_bucket, cog_data_prefix, year, local_output_dir=None)

In [13]:
# Check current cache status using the imported function
check_cache_status()

📊 Cache Status:
  - Directory: data_download/
  - Total files: 0
  - Total size: 0.00 GB


(0, 0)

# Useful links
<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/" style="color: blue; font-size: 20px;">drcs_activations OLD Directory</a> -- You can view old directory file structure here.

<a href="https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html" style="color: blue; font-size: 20px;">VEDA docs for file naming conventions</a> -- Helps for understanding why/how we name content.

# Process files

In [14]:
years = [2025]

In [15]:
keys

['sport-lis/rsm0-2m/20250217_0000_sport_lis_rsm0-2m_conus3km_float_wgs84.tif',
 'sport-lis/rsm0-2m/20250218_0000_sport_lis_rsm0-2m_conus3km_float_wgs84.tif',
 'sport-lis/rsm0-2m/20250219_0000_sport_lis_rsm0-2m_conus3km_float_wgs84.tif',
 'sport-lis/rsm0-2m/20250220_0000_sport_lis_rsm0-2m_conus3km_float_wgs84.tif',
 'sport-lis/rsm0-2m/20250221_0000_sport_lis_rsm0-2m_conus3km_float_wgs84.tif',
 'sport-lis/rsm0-2m/20250222_0000_sport_lis_rsm0-2m_conus3km_float_wgs84.tif',
 'sport-lis/rsm0-2m/20250223_0000_sport_lis_rsm0-2m_conus3km_float_wgs84.tif',
 'sport-lis/rsm0-2m/20250224_0000_sport_lis_rsm0-2m_conus3km_float_wgs84.tif',
 'sport-lis/rsm0-2m/20250225_0000_sport_lis_rsm0-2m_conus3km_float_wgs84.tif',
 'sport-lis/rsm0-2m/20250226_0000_sport_lis_rsm0-2m_conus3km_float_wgs84.tif',
 'sport-lis/rsm0-2m/20250227_0000_sport_lis_rsm0-2m_conus3km_float_wgs84.tif',
 'sport-lis/rsm0-2m/20250228_0000_sport_lis_rsm0-2m_conus3km_float_wgs84.tif',
 'sport-lis/rsm0-2m/20250301_0000_sport_lis_rsm0-2m_

In [16]:
# Define filename creator functions for different file types

#For NRT data, we should not have to change anything. But if we do, we can utilize this function

def create_cog_filename(f):
    """Create COG filename for SPORT-LIS files, moving datetime to end."""
    import re
    from pathlib import Path
    
    filename = Path(f).stem
    
    # Extract date and time from filename (YYYYMMDD_HHMM format)
    datetime_pattern = r'(\d{8})_(\d{4})'
    datetime_match = re.search(datetime_pattern, filename)
    
    if datetime_match:
        date_str = datetime_match.group(1)
        time_str = datetime_match.group(2)
        
        # Format as YYYY-MM-DDTHH:MM:SSZ
        year = date_str[:4]
        month = date_str[4:6]
        day = date_str[6:8]
        hour = time_str[:2]
        minute = time_str[2:4]
        
        formatted_datetime = f"{year}-{month}-{day}T{hour}:{minute}:00Z"
        
        # Remove the date and time portion from filename
        cleaned_filename = re.sub(r'\d{8}_\d{4}_', '', filename)
        
        # Build new filename
        cog_filename = f'{cleaned_filename}_{formatted_datetime}.tif'
    else:
        # Fallback
        cog_filename = f'{filename}.tif'
    
    return cog_filename

# Test examples:
test_files = [
 'sport-lis/vsm0-40cm/20250228_0000_sport_lis_vsm0-40cm_percentile_conus3km_float_wgs84.tif',
 'sport-lis/vsm0-40cm/20250301_0000_sport_lis_vsm0-40cm_percentile_conus3km_float_wgs84.tif',
 'sport-lis/vsm0-40cm/20250302_0000_sport_lis_vsm0-40cm_percentile_conus3km_float_wgs84.tif',
 'sport-lis/vsm0-40cm/20250303_0000_sport_lis_vsm0-40cm_percentile_conus3km_float_wgs84.tif'
]

for f in test_files:
    print(f"{f} -> {create_cog_filename(f)}")


sport-lis/vsm0-40cm/20250228_0000_sport_lis_vsm0-40cm_percentile_conus3km_float_wgs84.tif -> sport_lis_vsm0-40cm_percentile_conus3km_float_wgs84_2025-02-28T00:00:00Z.tif
sport-lis/vsm0-40cm/20250301_0000_sport_lis_vsm0-40cm_percentile_conus3km_float_wgs84.tif -> sport_lis_vsm0-40cm_percentile_conus3km_float_wgs84_2025-03-01T00:00:00Z.tif
sport-lis/vsm0-40cm/20250302_0000_sport_lis_vsm0-40cm_percentile_conus3km_float_wgs84.tif -> sport_lis_vsm0-40cm_percentile_conus3km_float_wgs84_2025-03-02T00:00:00Z.tif
sport-lis/vsm0-40cm/20250303_0000_sport_lis_vsm0-40cm_percentile_conus3km_float_wgs84.tif -> sport_lis_vsm0-40cm_percentile_conus3km_float_wgs84_2025-03-03T00:00:00Z.tif


In [ ]:
# Process cir files
import numpy as np
if keys:
    # Initialize combined results DataFrame
    all_files_processed = pd.DataFrame()
    for year in np.arange(2025,2026,1):
        print(year)
        
        #Only process a single years worth of files to make sure they save in the correct location
        year_keys = [i for i in keys if f'/{year}' in i]
        
        wm_results = process_file_batch(
            file_list=year_keys,  #CHANGE THIS
            config=add_year_to_config(config,year), #CHANGE THIS
            filename_creator_func=create_cog_filename, #CHANGE THIS
            s3_client=s3_client,
            processing_func=prepare_cogification,
            save_metadata=True,
            save_csv=True,
            verbose=True,
            BUCKET=BUCKET
        )
    all_files_processed = pd.concat([all_files_processed, wm_results], ignore_index=True)
    prepare_cogification
    # Print overall summary


2025
✅ Local output directory ready: output/rsm0-2m

[1/208] Processing: sport-lis/rsm0-2m/20250217_0000_sport_lis_rsm0-2m_conus3km_float_wgs84.tif
   Output filename: sport_lis_rsm0-2m_conus3km_float_wgs84_2025-02-17T00:00:00Z.tif
Old: False, New: False
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326...
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [COGIFY] Creating COG...
   [NODATA] Data type: float64
   [NODATA] Using nodata value -9999 for float64 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/nrt/sport-lis-cog/rsm0-2m/2025/sport_lis_rsm0-2m_conus3km_float_wgs84_2025-02-17T00:00:00Z.tif


In [41]:
# Display final results (for a single instance)
print(f"\n📊 Final Processing Results:")
print(f"Total files processed: {len(all_files_processed)}")
print(f"\nProcessed files DataFrame:")
all_files_processed


📊 Final Processing Results:
Total files processed: 207

Processed files DataFrame:


,file_name,COGs_created
0,sport-lis/rsm0-2m/20250217_0000_sport_lis_rsm0...,sport_lis_rsm0-2m_conus3km_float_wgs84_2025-02...
1,sport-lis/rsm0-2m/20250218_0000_sport_lis_rsm0...,sport_lis_rsm0-2m_conus3km_float_wgs84_2025-02...
2,sport-lis/rsm0-2m/20250219_0000_sport_lis_rsm0...,sport_lis_rsm0-2m_conus3km_float_wgs84_2025-02...
3,sport-lis/rsm0-2m/20250220_0000_sport_lis_rsm0...,sport_lis_rsm0-2m_conus3km_float_wgs84_2025-02...
4,sport-lis/rsm0-2m/20250221_0000_sport_lis_rsm0...,sport_lis_rsm0-2m_conus3km_float_wgs84_2025-02...
...,...,...
202,sport-lis/rsm0-2m/20250907_0000_sport_lis_rsm0...,sport_lis_rsm0-2m_conus3km_float_wgs84_2025-09...
203,sport-lis/rsm0-2m/20250908_0000_sport_lis_rsm0...,sport_lis_rsm0-2m_conus3km_float_wgs84_2025-09...
204,sport-lis/rsm0-2m/20250909_0000_sport_lis_rsm0...,sport_lis_rsm0-2m_conus3km_float_wgs84_2025-09...
205,sport-lis/rsm0-2m/20250910_0000_sport_lis_rsm0...,sport_lis_rsm0-2m_conus3km_float_wgs84_2025-09...


## Check STATUS of file conversion and upload

<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/" style="color: blue; font-size: 20px;">Disasters Bucket</a> -- You can view that the files actually made it to their correct destination. 
